## PREPROCESAMIENTO DE DATASET SINTETICO POR PASOS

Generación de dataset sintetico (para probar limpieza)

In [2]:
import sys; sys.path.append('../..')

from src.utils.synthetic_generator import SyntheticDataGenerator
from src.utils.constants import SYNTHETIC_DATA_PATH

print("=" * 70)
print("GENERACIÓN DE DATASET SINTÉTICO")
print("=" * 70)

generator = SyntheticDataGenerator(n_samples=300, seed=42)

# Generar dataset con problemas
df_synthetic = generator.generate_problematic_dataset(
    problem_types=['missing_codes', 'out_of_range', 'nans', 'outliers']
)

# Mostrar resumen
generator.print_summary()

# Guardar
synthetic_path = f"{SYNTHETIC_DATA_PATH}/problematic_test.dta"
generator.save_dataset(str(synthetic_path))

print(f"\n✓ Dataset sintético generado y listo para usar")
print("=" * 70)

GENERACIÓN DE DATASET SINTÉTICO

RESUMEN DEL DATASET SINTÉTICO
Dimensiones: 300 registros × 11 variables

Valores faltantes (NaN):
  edre              36 ( 12.0%)
  q10inc            36 ( 12.0%)
  ocupoit           36 ( 12.0%)

Códigos especiales LAPOP:
  No sabe               211
  No responde           226
  Inaplicable           251
✓ Dataset sintético guardado en: C:\electocluster\data\synthetic\lapop_bolivia_2023.dta/problematic_test.dta

✓ Dataset sintético generado y listo para usar


PASO 1 - Carga del dataset (loader.py)

In [4]:
import sys; sys.path.append('../..')

from src.preprocessing.loader import DatasetLoader

print("=" * 70)
print("BLOQUE 1: TEST DATASETLOADER")
print("=" * 70)

loader = DatasetLoader(synthetic_path)

df = loader.load()

# Mostrar resumen
print(f"\n✓ Dataset cargado exitosamente")
print(f"  Dimensiones: {df.shape[0]} registros × {df.shape[1]} variables")
print(f"\nPrimeras 5 filas:")
print(df.head())

print(f"\nVariables seleccionadas:")
for col in df.columns:
    print(f"  - {col}")

print(f"\nMissings por variable:")
missing_counts = df.isnull().sum()
for var, count in missing_counts.items():
    if count > 0:
        pct = (count / len(df)) * 100
        print(f"  {var:15} {count:4} ({pct:5.1f}%)")

print("\n" + "=" * 70)

BLOQUE 1: TEST DATASETLOADER
Cargando dataset desde: C:\electocluster\data\synthetic\lapop_bolivia_2023.dta/problematic_test.dta
✓ Dataset cargado: 300 registros × 11 variables

✓ Dataset cargado exitosamente
  Dimensiones: 300 registros × 11 variables

Primeras 5 filas:
       q2  q12cn      edre    q10inc    etid  ur  ocupoit  q1tc_r    q11n  \
0      69      3  988888.0    1013.0       5   1      NaN       2       1   
1  999999      7       6.0  999999.0       1   2      NaN  888888       4   
2      78      2  999999.0    1005.0       1   1      4.0       1       4   
3      38      2       5.0       NaN  888888   1      5.0       1       1   
4      41      7  888888.0  999999.0       1   1      4.0  988888  888888   

   boletidnew    q3cn  
0           2      11  
1      988888       2  
2           1  888888  
3           2  988888  
4           2  988888  

Variables seleccionadas:
  - q2
  - q12cn
  - edre
  - q10inc
  - etid
  - ur
  - ocupoit
  - q1tc_r
  - q11n
  - boleti

PASO 2 - Limpieza del dataset (cleaner.py)

In [5]:
from src.preprocessing.cleaner import DataCleaner

print("=" * 70)
print("BLOQUE 2: TEST DATACLEANER")
print("=" * 70)

# Usar el df del BLOQUE 1
print(f"\nDataset antes de limpieza:")
print(f"  Dimensiones: {df.shape}")
print(f"  Missings totales: {df.isnull().sum().sum()}")

cleaner = DataCleaner(df.copy())

# Limpieza paso a paso
print("\n--- Validando rangos ---")
cleaner.validate_ranges()

print("\n--- Imputando valores faltantes ---")
cleaner.handle_missing_values()

print("\n--- Detectando outliers ---")
outliers = cleaner.detect_outliers(method='iqr', threshold=1.5)

# Obtener dataset limpio
df_clean = df

print(f"\nDataset después de limpieza:")
print(f"  Dimensiones: {df_clean.shape}")
print(f"  Missings totales: {df_clean.isnull().sum().sum()}")

print(f"\nPrimeras 5 filas del dataset limpio:")
print(df_clean.head())

print("\n" + "=" * 70)

BLOQUE 2: TEST DATACLEANER

Dataset antes de limpieza:
  Dimensiones: (300, 11)
  Missings totales: 108

--- Validando rangos ---
Validando rangos de variables...
  ⚠ q2: 72 valores fuera de rango [18, 120] → NaN
  ⚠ edre: 57 valores fuera de rango [0, 6] → NaN
  ⚠ q10inc: 58 valores fuera de rango [1001, 1015] → NaN
  ⚠ etid: 68 valores fuera de rango [1, 7] → NaN
  ⚠ boletidnew: 63 valores fuera de rango [1, 2] → NaN
  ⚠ ur: 68 valores fuera de rango [1, 2] → NaN
  ⚠ ocupoit: 58 valores fuera de rango [1, 10] → NaN
  ⚠ q1tc_r: 64 valores fuera de rango [1, 3] → NaN
  ⚠ q3cn: 70 valores fuera de rango [1, 77] → NaN
  ⚠ q12cn: 71 valores fuera de rango [1, 25] → NaN
  ⚠ q11n: 68 valores fuera de rango [1, 6] → NaN
✓ Validación de rangos completada

--- Imputando valores faltantes ---
Imputando valores faltantes...
  ✓ q2: 72 missings imputados con mediana (50.0)
  ✓ edre: 93 missings imputados con moda (1.0)
  ✓ q10inc: 94 missings imputados con moda (1015.0)
  ✓ etid: 68 missings impu

PASO 3 - Transformación de datos del dataset (transformer.py)

In [6]:

from src.preprocessing.transformer import DataTransformer

print("=" * 70)
print("BLOQUE 3: TEST DATATRANSFORMER")
print("=" * 70)

# Usar df_clean del BLOQUE 2
print(f"\nDataset antes de transformación:")
print(f"  Dimensiones: {df_clean.shape}")
print(df_clean.dtypes)

#Aqui no se hace feature engineering, solo transformación de variables
#   Esto debido a que el dataset sintético no tiene variables de riqueza ni cívicas. 

transformer = DataTransformer(df_clean.copy())

# Ejecutar transformaciones
print(f"\n--- Normalizando variables numéricas ---")
df_transformed = transformer.normalize_numeric_features(method='minmax')

print(f"\n--- Codificando variables categóricas ---")
#df_encoded = transformer.encode_categorical_features(method='ordinal')

print(f"\nDataset después de transformación:")
print(f"  Dimensiones: {df_transformed.shape}")
print(df_transformed.dtypes)

print(f"\nPrimeras 5 filas del dataset transformado:")
print(df_transformed.head())

print(f"\nEstadísticas descriptivas:")
print(df_transformed.describe())

print("\n" + "=" * 70)
print("✓ TODAS LAS CLASES PROBADAS EXITOSAMENTE")
print("=" * 70)

BLOQUE 3: TEST DATATRANSFORMER

Dataset antes de transformación:
  Dimensiones: (300, 11)
q2              int32
q12cn           int32
edre          float64
q10inc        float64
etid            int32
ur              int32
ocupoit       float64
q1tc_r          int32
q11n            int32
boletidnew      int32
q3cn            int32
dtype: object

--- Normalizando variables numéricas ---
Normalizando variables numéricas con método 'minmax'...
  ✓ q2 normalizado
  ✓ q12cn normalizado
✓ Normalización completada (2 variables)

--- Codificando variables categóricas ---

Dataset después de transformación:
  Dimensiones: (300, 11)
q2            float64
q12cn         float64
edre          float64
q10inc        float64
etid            int32
ur              int32
ocupoit       float64
q1tc_r          int32
q11n            int32
boletidnew      int32
q3cn            int32
dtype: object

Primeras 5 filas del dataset transformado:
         q2     q12cn      edre    q10inc    etid  ur  ocupoit  q1tc_r